In [1]:
!pip install pandas
!pip install tensorflow
!pip install scikit-optimize
!pip install dask[complete]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 57.2 MB/s eta 0:00:00


In [2]:
import dask
from dask.distributed import Client, LocalCluster

n_workers = 2
thread_per_worker = 1
dask.config.set(scheduler='threads', num_of_workers=n_workers, threads_per_worker=thread_per_worker)
cluster = LocalCluster(n_workers=n_workers, threads_per_worker=thread_per_worker, dashboard_address=':8888')
client = Client(cluster)
print(f'{cluster.dashboard_link}')


INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:43591
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:8888/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:43213'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:36703'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:43161 name: 0
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:43161
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:45952
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:40177 name: 1
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:40177
INFO:distributed.core:Starting established connection to tcp://127

http://127.0.0.1:8888/status


In [3]:
!pip install pyngrok
from pyngrok import ngrok, conf
import getpass
from urllib.parse import urlparse

print("Enter your authtoken, which can be copied "
"from https://dashboard.ngrok.com/get-started/your-authtoken")
conf.get_default().auth_token = '2je6T7K1ne0d0gnWHzjRmHcvvaD_pecT3awMJ4zGgCiyFZn6'

ui_port = urlparse(client.dashboard_link).port
public_url = ngrok.connect(ui_port).public_url
print(f" * ngrok tunnel \"{public_url}\" -> \"http://127.0.0.1:{ui_port}\"")

Enter your authtoken, which can be copied from https://dashboard.ngrok.com/get-started/your-authtoken
 * ngrok tunnel "https://7edd3a43a0d1.ngrok-free.app" -> "http://127.0.0.1:8888"


In [4]:
import gdown
import zipfile
import os
# UCF 4 class link
# shared_link = "https://drive.google.com/file/d/1HyXsNuW-P2yYEdtnTwU70CYCAFa-3Jse/view?usp=sharing"
# UCF 10 class link
shared_link = "https://drive.google.com/file/d/14z32aNr6m86q3TaV2chAoJG8cYfsF8nO/view?usp=sharing"
# Extract the file ID from the shared link
file_id = shared_link.split('/d/')[1].split('/')[0]
download_url = f"https://drive.google.com/uc?id={file_id}"
# Define the output path for the downloaded ZIP file
output_zip = "file.zip"
# Download the file
gdown.download(download_url, output_zip, quiet=False)
# Extract the ZIP file
output_folder = "extracted_files"
os.makedirs(output_folder, exist_ok=True)
with zipfile.ZipFile(output_zip, 'r') as zip_ref:
    zip_ref.extractall(output_folder)
print(f"Files extracted to: {output_folder}")
# List the extracted files
for root, dirs, files in os.walk(output_folder):
    for file in files:
        print(os.path.join(root, file))

Downloading...
From (original): https://drive.google.com/uc?id=14z32aNr6m86q3TaV2chAoJG8cYfsF8nO
From (redirected): https://drive.google.com/uc?id=14z32aNr6m86q3TaV2chAoJG8cYfsF8nO&confirm=t&uuid=5a05510f-ba61-47bc-88f7-96a0882caeec
To: /content/file.zip
100%|██████████| 732M/732M [00:31<00:00, 23.4MB/s]


Files extracted to: extracted_files
extracted_files/PushUps/v_PushUps_g11_c01.avi
extracted_files/PushUps/v_PushUps_g03_c03.avi
extracted_files/PushUps/v_PushUps_g13_c01.avi
extracted_files/PushUps/v_PushUps_g02_c03.avi
extracted_files/PushUps/v_PushUps_g19_c02.avi
extracted_files/PushUps/v_PushUps_g25_c02.avi
extracted_files/PushUps/v_PushUps_g11_c04.avi
extracted_files/PushUps/v_PushUps_g14_c03.avi
extracted_files/PushUps/v_PushUps_g26_c01.avi
extracted_files/PushUps/v_PushUps_g04_c04.avi
extracted_files/PushUps/v_PushUps_g07_c02.avi
extracted_files/PushUps/v_PushUps_g16_c02.avi
extracted_files/PushUps/v_PushUps_g08_c03.avi
extracted_files/PushUps/v_PushUps_g07_c01.avi
extracted_files/PushUps/v_PushUps_g14_c01.avi
extracted_files/PushUps/v_PushUps_g08_c02.avi
extracted_files/PushUps/v_PushUps_g07_c03.avi
extracted_files/PushUps/v_PushUps_g24_c01.avi
extracted_files/PushUps/v_PushUps_g17_c04.avi
extracted_files/PushUps/v_PushUps_g06_c04.avi
extracted_files/PushUps/v_PushUps_g01_c02.av

In [5]:
import math
import random
from abc import abstractmethod, ABC
import heapq
import numpy as np
import dask.array as da

count = -1

def generate_synopsis_id():
    global count
    count += 1
    return count


class Synopsis(ABC):

    def __init__(self, key_index=None, value_index=None):
        self.synopsis_id = generate_synopsis_id()
        self.key_index = None
        self.value_index = None

    def get_synopsis_id(self):
        return self.synopsis_id

    def set_synopsis_id(self, synopsis_id):
        self.synopsis_id = synopsis_id

    def set_key_index(self, key_index):
        self.key_index = key_index

    def get_key_index(self):
        return self.key_index

    def set_value_index(self, value_index):
        self.value_index = value_index

    def get_value_index(self):
        return self.value_index

    @abstractmethod
    def add(self, key_index, value_index):
        pass

    @abstractmethod
    def estimate(self, key_index):
        pass

    @abstractmethod
    def merge(self, other_synopsis):
        pass

    def operation_mode_add(self, obj):
        # This seems to be specific to the image, and its purpose is unclear.
        # You'll need to provide more context or its intended behavior.
        raise NotImplementedError

    def get_hash_count(self):
        return self.hash_count


class WeightedPrioritySampler(Synopsis):
    """
    A streaming sampler that keeps up to k items with the smallest
    'priority', where priority = -log(U) / weight.
    """

    def __init__(self, k, seed=None):
        self.k = k
        self.heap = []  # will store tuples: (neg_priority, value, weight)
        self.random_state = random.Random(seed)

    def add_serial(self, value, weight=1.0):
        if weight <= 0:
            return
        u = self.random_state.random()
        if u <= 0:
            return
        priority = -math.log(u) / weight
        neg_priority = -priority

        if len(self.heap) < self.k:
            heapq.heappush(self.heap, (neg_priority, value, weight))
        else:
            if neg_priority > self.heap[0][0]:
                heapq.heapreplace(self.heap, (neg_priority, value, weight))

    def add(self, chunk):
        """
        Add chunks of items (values and weights) to the sampler.
        Arguments:
            values: List or array of values.
            weights: List or array of weights (same size as values).
        """

        chunk_shape = chunk.shape
        print(f"chunk_shape: {chunk_shape}")

        # Separate values and weights from the data
        values, weights = np.array(chunk).T

        if np.any(weights <= 0):
            print("negative weights found !!!")
            return

        # Generate random U values
        u_values = np.random.random(size=values.shape)

        if np.any(u_values <= 0):
            print("negative u_values found !!!")
            return

        # Compute priorities
        priorities = -np.log(u_values) / weights
        neg_priorities = -priorities

        # Combine into a single array: (neg_priority, value, weight)
        combined = np.stack([neg_priorities, values, weights], axis=1)

        # Sort combined array by neg_priority (column 0) and select top k
        top_k_indices = np.argsort(-combined[:, 0])[:self.k]  # Sort by neg_priority desc
        print(f"top_k_indices: {top_k_indices}")

        top_k = combined[top_k_indices]  # Get top k rows
        print(f"top k: {top_k.shape}")

        padding = da.zeros((chunk_shape[0], top_k.shape[1]), chunks="auto")

        padding[0:top_k.shape[0], :] = top_k[0:top_k.shape[0], :]

        print(f"padding: {padding}")

        self.heap = top_k

        return top_k

    """
    def merge(self, other):
        for neg_priority, value, weight in other.heap:
            if len(self.heap) < self.k:
               heapq.heappush(self.heap, (neg_priority, value, weight))
            else:
               if neg_priority > self.heap[0][0]:
                  heapq.heapreplace(self.heap, (neg_priority, value, weight))
    """

    def merge(self, tables):
        """
        Merge multiple tables, keeping only the top k rows based on the first column (priority).

        Arguments:
            tables: List of 2D NumPy arrays to merge.
            k: Number of top rows to keep based on the first column (priority).

        Returns:
            2D NumPy array with the top k rows.
        """
        # Concatenate all tables
        # Sort by the first column (priority) in descending order
        merged_table = da.vstack(tables)
        merged_table_top_k = merged_table[da.argtopk(-merged_table[:, 0], self.k)[::-1]]
        # print(f"merged_table_top_k: {merged_table_top_k}")

        self.heap = merged_table_top_k

        return self

    """
    def estimate(self):
        total_weight = 0.0
        weighted_sum = 0.0
        for neg_priority, value, weight in self.heap:
            total_weight += weight
            weighted_sum += value * weight
        if total_weight == 0.0:
            return 0.0
        return weighted_sum / total_weight
    """

    def estimate(self):
        total_weight = da.sum(self.heap[:, 2]).compute()
        # print(f"weights: {self.heap[:, 2]}")
        # print(f"values: {self.heap[:, 1]}")
        values_weights = self.heap[:, 1] * self.heap[:, 2]
        # print(f"values * weights: {values_weights}")
        weighted_sum = da.sum(values_weights).compute()
        if total_weight == 0.0:
            return 0.0
        return weighted_sum / total_weight

    @staticmethod
    def build_sampler_for_partition(partition_data, k, seed):
        """
        Given an iterable of (value, weight) pairs in this partition,
        build and return a WeightedPrioritySampler of size k.
        """
        sampler = WeightedPrioritySampler(k, seed=seed)
        for (value, weight) in partition_data:
            sampler.add(value, weight)
        return sampler


class PrioritySampler(WeightedPrioritySampler):
    """
    A streaming sampler that keeps up to k items with the smallest
    random priority (uniform sampling).

    We store (neg_priority, item) in a min-heap (by neg_priority),
    which simulates a max-heap for actual priority.
    """

    def __init__(self, k, seed=None):
        """
        Parameters
        ----------
        k : int
            Maximum number of items to sample.
        seed : int or None, optional
            Random seed for reproducible priorities.
        """
        self.k = k
        # self.heap = []  # will store (neg_priority, item)
        self.random_state = random.Random(seed)

    def add(self, item):
        """
        Add a single item with a random priority.
        """
        # Generate a priority in [0, 1).
        priority = self.random_state.random()
        # We store negative priority, so the item with the largest
        # actual priority is on top of the min-heap (heap[0]).
        neg_priority = -priority

        if len(self.heap) < self.k:
            # If the heap is not full, just push the new item.
            heapq.heappush(self.heap, (neg_priority, item))
        else:
            # If the heap is full, compare with the top item.
            # If the new item has a smaller priority => bigger neg_priority => replace
            if neg_priority > self.heap[0][0]:
                heapq.heapreplace(self.heap, (neg_priority, item))

    def add_chunk(self, chunk):
        """
        Add chunks of items to the sampler.
        """
        # np.set_printoptions(formatter={'float': lambda x: "{0:0.3f}".format(x)})

        chunk = chunk[:, 0]
        chunk_shape = chunk.shape

        #print(f"chunk: {chunk}")

        #if np.any(chunk <= 0):
        #print("negative chunk found !!!")
        #return

        priorities = da.random.random(size=chunk_shape)

        neg_priorities = -priorities

        # Combine into a single array: (neg_priority, value, weight)
        combined = da.stack([neg_priorities, chunk], axis=1)
        # print(f"combined: {combined.shape}")
        # Sort combined array by neg_priority (column 0) and select top k
        top_k_indices = da.argtopk(combined[:, 0], self.k)  # Sort by neg_priority desc
        #print(f"top_k_indices: {top_k_indices}")
        #print(f"combined: {combined.shape}")
        # top_k = combined[top_k_indices]  # Get top k rows
        # #print(f"top_k_shape: {top_k.shape[0]} & {top_k.shape[1]}")
        # #padding = da.zeros((chunk_shape[0], top_k.shape[1]), chunks = "auto")
        #
        # #padding[0:top_k.shape[0], :] = top_k[0:top_k.shape[0], :]
        #
        # self.heap = top_k
        #print(f"top k: {top_k.shape}")
        #print(f"padding: {padding}")

        return top_k_indices

    def reservoir_priority_sampling(self, all_data, k, divided_by):
        num_elements = len(all_data)
        self.k = int(k) // divided_by
        zeros = da.zeros((num_elements), chunks=(num_elements // divided_by))

        # Apply rechunk to control the chunk size
        all_data_indexes = da.arange(num_elements, chunks=(num_elements // divided_by))
        data = da.stack([all_data_indexes, zeros], axis=1)
        data = data.rechunk(num_elements // divided_by, 2)  # Control the chunk size here
        top_ks_splitted = data.map_blocks(self.add_chunk, dtype=all_data.dtype)
        top_ks_splitted = top_ks_splitted.compute()
        # Print the overall length

        array_version = np.array(top_ks_splitted).flatten()
        # merged_PS = self.merge(top_ks_splitted)
        # sampled_indexes = self.estimate()
        return all_data[array_version].compute()

    """
    def merge(self, other):
        #Merge another PrioritySampler into this one, preserving only
        #the top k smallest priorities overall.

        for neg_priority, item in other.heap:
            if len(self.heap) < self.k:
                heapq.heappush(self.heap, (neg_priority, item))
            else:
                if neg_priority > self.heap[0][0]:
                    heapq.heapreplace(self.heap, (neg_priority, item))
    """

    def estimate(self):
        # if not self.heap:
        #     return da.array([])  # Return an empty Dask array if heap is empty
        #
        # heap_array = da.array(self.heap)  # Convert list to Dask array
        # return_array = da.round(heap_array[:, 1]).astype(int)
        # return return_array
        # print(f"values: {da.array(self.heap[:, 1]).compute()}")
        return_array = da.round(self.heap[:, 1]).astype(int)
        return return_array

    def get_sample(self):
        """
        Return the sampled items (in no particular order).
        """
        return [item for (neg_priority, item) in self.heap]

    @staticmethod
    def build_sampler_for_partition(partition_data, k, seed):
        """
        Create a PrioritySampler for this partition's data.
        partition_data is an iterable of items (all unweighted).
        """
        sampler = PrioritySampler(k=k, seed=seed)
        for item in partition_data:
            sampler.add(item)
        return sampler



class PartialDFTAccumulator:
    def __init__(self, dft_sum=None):
        self.dft_sum = dft_sum  # Holds the summed DFT (complex array)
        self.n = None  # Length of each partial signal

    def add(self, chunk):
        """
        Compute the DFT of this chunk (1D array) and store it (by summation).
        """
        if self.n is None or self.n == 0:
            self.n = len(chunk)
        elif len(chunk) != self.n:
            raise ValueError("All chunks must have the same length to sum DFTs.")

        partial_dft = np.fft.fft(chunk)

        if self.dft_sum is None:
            self.dft_sum = partial_dft
        else:
            self.dft_sum += partial_dft
        return self

    def merge(self, *others):
        """
        Merge multiple PartialDFTAccumulators by summing their DFTs.
        """
        for other in others:
            if other.dft_sum is None:
                continue  # Skip empty accumulators

        if self.dft_sum is None:
            self.dft_sum = others[0].dft_sum
            self.n = others[0].n
        else:
            other_dfts = [other.dft_sum for other in others if other.dft_sum is not None]
            self.dft_sum += np.sum(other_dfts, axis=0)

        return self

    def estimate(self):
        """
        Return the final merged DFT.
        """
        if self.dft_sum is None:
            return None  # no data
        return self.dft_sum

    @staticmethod
    def build_accumulator_for_partition(chunk):
        """
        Convert a chunk into a PartialDFTAccumulator.
        """
        acc = PartialDFTAccumulator()
        acc.add(chunk)
        return acc

    @staticmethod
    def merge_many_accumulators(*accumulators):
        """
        Merge multiple PartialDFTAccumulators into one.
        """
        return accumulators[0].merge(*accumulators[1:])


def merge_samplers(sampler_a, sampler_b):
    """
    Merge sampler_b into sampler_a and return sampler_a.
    """
    sampler_a.merge(sampler_b)
    return sampler_a


In [6]:
import os
import math
import time
import json
import numpy as np
from sklearn.model_selection import train_test_split
import cv2
import sys

# Producer part
def error_callback(exc):
  raise Exception('Error while sendig data to kafka: {0}'.format(str(exc)))


def frames_extraction(video_path):
  # Store the video frames
  frames_list = []

  # Read the video
  video_reader = cv2.VideoCapture(video_path)

  # Get total number of frames (of this video)
  video_frames_count = int(video_reader.get(cv2.CAP_PROP_FRAME_COUNT))

  # Calculate the interval after which frames will be stored (the step)
  skip_frames_window = max(int(video_frames_count/SEQUENCE_LENGTH), 1)

  # Iterate over video-frames
  for frame_counter in  range(SEQUENCE_LENGTH):
    # Adjust the pointer of current frame
    video_reader.set(cv2.CAP_PROP_POS_FRAMES, frame_counter * skip_frames_window)

    # Read the corresponding frame
    success, frame = video_reader.read()

    if not success:
      break

    # Resize-normalize the frame and save it to the corresponding list
    resized_frame = cv2.resize(frame, (64, 64))
    frames_list.append(resized_frame)
    # normalized_frame = resized_frame / 255
    # frames_list.append(normalized_frame)

  video_reader.release()

  return frames_list


def create_dataset():
  if len(CLASSES_LIST) == 4:
    data_dir = "/content/extracted_files/UCF4"
  else:
    data_dir = "/content/extracted_files"
  # Lists that contain the extracted features, the labels and the paths of the videos
  features = []
  labels = []
  video_files_paths = []

  # Iterate through all (selected) classes
  for class_index, class_name in enumerate(CLASSES_LIST):
    print(f'Extracting Data of Class: {class_name}')

    # Get the videos that are contained in each (selected) class
    files_list = os.listdir(os.path.join(data_dir, class_name))
    for file_name in files_list:
      video_file_path = os.path.join(data_dir, class_name, file_name)
      frames = frames_extraction(video_file_path)
      if len(frames) == SEQUENCE_LENGTH:
        features.append(frames)
        labels.append(class_index)
        video_files_paths.append(video_file_path)

  # Convert lists to numpy arrays
  # features = np.repeat(features, 70, axis=0)
  # print("repeated features")
  # labels = np.repeat(labels, 70, axis=-1)
  # print("repeated labels")
  features = np.asarray(features)
  labels = np.asarray(labels)
  return features, labels, video_files_paths


try:
  with open('config_video.json') as json_file:
    config = json.load(json_file)
except:
  print("config_video.json not found")
  exit()
  args = sys.argv[1:]
SEQUENCE_LENGTH = config['sequence_length']
tmp_filter_train = config['stream_batch_train'] * SEQUENCE_LENGTH
tmp_filter_test = config['stream_batch_test'] * SEQUENCE_LENGTH
CLASSES_LIST = config['classes_list']
features, labels, video_files_paths = create_dataset()
features_train, features_test, labels_train, labels_test = train_test_split(features, labels, test_size=0.25, shuffle=True)
received_images_reshaped = features_train
received_labels_decoded = labels_train
received_images_reshaped_test = features_test
received_labels_decoded_test = labels_test

Extracting Data of Class: WalkingWithDog
Extracting Data of Class: HorseRace
Extracting Data of Class: Diving
Extracting Data of Class: PushUps
Extracting Data of Class: Fencing
Extracting Data of Class: Punch
Extracting Data of Class: Skiing
Extracting Data of Class: MilitaryParade
Extracting Data of Class: Rowing
Extracting Data of Class: Billiards


In [7]:
tmp1 = received_images_reshaped
tmp2 = received_labels_decoded
tmp3 = received_images_reshaped_test
tmp4 = received_labels_decoded_test

In [8]:
import copy
x_train = copy.deepcopy(tmp1)
y_train = copy.deepcopy(tmp2)
x_test = copy.deepcopy(tmp3)
y_test = copy.deepcopy(tmp4)
sampling_method_id = 2

In [9]:
import dask.array as da
print(f"train images len: {x_train.shape}")
print(f"train labels len: {y_train.shape}")
y_train = y_train.reshape(-1, 1)
divided_by = 4

rep_factor = 1

if sampling_method_id == 2:
  x_train = da.from_array(np.tile(x_train, (rep_factor,1,1,1,1)), chunks=((len(x_train) * rep_factor) // divided_by, 20, 64 ,64 ,3))  # You can adjust the chunk size as needed
  y_train = da.from_array(np.tile(y_train, (rep_factor,1)), chunks=((len(y_train) * rep_factor) // divided_by, 1))  # You can adjust the chunk size as needed
else:
  x_train = np.tile(x_train, (rep_factor,1,1,1))  # You can adjust the chunk size as needed
  y_train = np.tile(y_train, (rep_factor,1))  # You can adjust the chunk size as needed

print(f"train labels: {y_train}")
print(f"train images: {x_train}")

# Compute unique labels
unique_class_labels = da.unique(y_train).compute()
print(x_train.shape)
print(y_train.shape)
#unique_class_labels = np.unique(train_labels_all)
# x_train, x_test = x_train / 255.0, x_test / 255.0


train images len: (1003, 20, 64, 64, 3)
train labels len: (1003,)
train labels: dask.array<array, shape=(1003, 1), dtype=int64, chunksize=(250, 1), chunktype=numpy.ndarray>
train images: dask.array<array, shape=(1003, 20, 64, 64, 3), dtype=uint8, chunksize=(250, 20, 64, 64, 3), chunktype=numpy.ndarray>
(1003, 20, 64, 64, 3)
(1003, 1)


In [21]:
from re import I
import gc
import json
import multiprocessing
import pickle
import socket
import sys
import random
import math
from datetime import datetime
import time
import json
from multiprocessing import Queue
import pandas as pd
import tensorflow as tf
import numpy as np
from matplotlib import pyplot as plt
from tensorflow.keras.datasets import fashion_mnist, mnist, cifar10, cifar100
from tensorflow.keras import layers, models
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import Adam
from skopt import gbrt_minimize, gp_minimize
from skopt.utils import use_named_args
from skopt.space import Real, Categorical, Integer
from tensorflow.python.keras import backend as K
import matplotlib.pyplot as plt
import struct
import dask.array as da
from threading import Thread, Event


try:
    with open('config_video.json') as json_file:
        config = json.load(json_file)
except:
    print("config_video.json not found")
    exit()

call_counter = 0
SEQUENCE_LENGTH = config['sequence_length']
tmp_filter_train = config['bo_data_train'] * SEQUENCE_LENGTH
tmp_filter_test = config['bo_data_test'] * SEQUENCE_LENGTH
size_of_batch = config['size_of_batch']
lr = config['lr']
# size_of_batch_low = config['size_of_batch_low']
# size_of_batch_high = config['size_of_batch_high']
# lr_low = config['lr_low']
# lr_high = config['lr_high']
num_of_conv_layers_low = config['num_of_conv_layers_low']
num_of_conv_layers_high = config['num_of_conv_layers_high']
num_of_pool_layers_low = config['num_of_pool_layers_low']
num_of_pool_layers_high = config['num_of_pool_layers_high']
num_of_dense_layers_low = config['num_of_dense_layers_low']
num_of_dense_layers_high = config['num_of_dense_layers_high']
num_of_lstm_layers_low = config['num_of_lstm_layers_low']
num_of_lstm_layers_high = config['num_of_lstm_layers_high']
num_of_gru_layers_low = config['num_of_gru_layers_low']
num_of_gru_layers_high = config['num_of_gru_layers_high']
num_of_rnn_layers_low = config['num_of_rnn_layers_low']
num_of_rnn_layers_high = config['num_of_rnn_layers_high']
num_of_dask_workers_low = config['num_of_dask_workers_low']
num_of_dask_workers_high = config['num_of_dask_workers_high']
sample_size_low = config['sample_size_low']
sample_size_high = config['sample_size_high']
num_of_epochs_low = config['num_of_epochs_low']
num_of_epochs_high = config['num_of_epochs_high']
sampling_method_id = 2
acquisition_f = config['acquisition_f']
theta_parameter = config['theta_parameter']
lamda_acc = config['lamda_acc']
bo_call_number = config['bo_call_number']
DATASET_SHAPE = [64, 64, 3]
UNIQUE_CLASS_LABELS = range(len(config['classes_list']))
# Percentage of the dataset that will be used for testing
perc_test = 1

# tunnel of 65433
streamlit_tunnel_addr = "7.tcp.eu.ngrok.io"
streamlit_tunnel_port = 16691

# tunnel of 65435
streamlit_live_tunnel_addr = "2.tcp.eu.ngrok.io"
streamlit_live_tunnel_port = 12121

# tunnel of 65436
sbto_run_tunnel_addr = "4.tcp.eu.ngrok.io"
sbto_run_tunnel_port = 19118

extra_results = []

dimensions = []
dimension_names = []
default_parameters = []
column_names = []
dim_sample_size = Real(low=sample_size_low, high=sample_size_high, name='sample_size')
dimensions.append(dim_sample_size)
dimension_names.append('sample_size')
column_names.append('Sample Size')
default_parameters.append(0.15)
dim_epochs_number = Integer(low=num_of_epochs_low, high=num_of_epochs_high, name='epochs_number')
dimensions.append(dim_epochs_number)
dimension_names.append('epochs_number')
column_names.append('Epochs')
default_parameters.append(5)
dim_conv_number = Integer(low=num_of_conv_layers_low, high=num_of_conv_layers_high, name='conv_number')
dimensions.append(dim_conv_number)
dimension_names.append('conv_number')
column_names.append('Conv')
default_parameters.append(2)
dim_pool_number = Integer(low=num_of_pool_layers_low, high=num_of_pool_layers_high, name='pool_number')
dimensions.append(dim_pool_number)
dimension_names.append('pool_number')
column_names.append('Pool')
default_parameters.append(2)
if num_of_lstm_layers_low != num_of_lstm_layers_high:
  dim_lstm_number = Integer(low=num_of_lstm_layers_low, high=num_of_lstm_layers_high, name='lstm_number')
  dimensions.append(dim_lstm_number)
  dimension_names.append('lstm_number')
  column_names.append('LSTM')
  default_parameters.append(1)
if num_of_gru_layers_low != num_of_gru_layers_high:
  dim_gru_number = Integer(low=num_of_gru_layers_low, high=num_of_gru_layers_high, name='gru_number')
  dimensions.append(dim_gru_number)
  dimension_names.append('gru_number')
  column_names.append('GRU')
  default_parameters.append(0)
if num_of_rnn_layers_low != num_of_rnn_layers_high:
  dim_rnn_number = Integer(low=num_of_rnn_layers_low, high=num_of_rnn_layers_high, name='rnn_number')
  dimensions.append(dim_rnn_number)
  dimension_names.append('rnn_number')
  column_names.append('RNN')
  default_parameters.append(0)
dim_dense_number = Integer(low=num_of_dense_layers_low, high=num_of_dense_layers_high, name='dense_number')
dimensions.append(dim_dense_number)
dimension_names.append('dense_number')
column_names.append('Dense')
default_parameters.append(2)
dim_dask_workers = Integer(low=num_of_dask_workers_low, high=num_of_dask_workers_high, name='dask_workers')
dimensions.append(dim_dask_workers)
dimension_names.append('dask_workers')
column_names.append('Dask Workers')
default_parameters.append(4)
# dim_lr = Real(low=lr_low, high=lr_high, name='learning_rate')
# dim_batch_size = Integer(low=size_of_batch_low, high=size_of_batch_high, name='batch_size')
# dimensions = [dim_sample_size,
#               dim_epochs_number,
#               dim_conv_number,
#               dim_pool_number,
#               dim_lstm_number,
#               dim_gru_number,
#               dim_rnn_number,
#               dim_dense_number
#              ]
# default_parameters = [0.99, 4, 2, 2, 1, 0, 0, 1]
# default_parameters = [(sample_size_low + sample_size_high) / 2,
#                      (num_of_epochs_low + num_of_epochs_high) // 2,
#                      (num_of_conv_layers_low + num_of_conv_layers_high) // 2,
#                      (num_of_pool_layers_low + num_of_pool_layers_high) // 2,
#                      (num_of_lstm_layers_low + num_of_lstm_layers_high) // 2,
#                      (num_of_gru_layers_low + num_of_gru_layers_high) // 2,
#                      (num_of_rnn_layers_low + num_of_rnn_layers_high) // 2,
#                      (num_of_dense_layers_low + num_of_dense_layers_high) // 2]


CONV_PADDING = 'same'
MAX_POOL_PADDING = 'same'
CONV_NEURONS_CONST = 16
CONV_NEURONS_BOUND = 64
DENSE_NEURONS_CONST = 128
DENSE_NEURONS_BOUND = 32
UNITS_CONST = 32
UNITS_BOUND = 32
streamlit_live_socket = None

def start_controller():
    """
    Initialize the socket and listen for keyboard input
    """
    # Open config file and get the desired port for socket communication
    try:
        with open('config_video.json') as json_file:
            config = json.load(json_file)
    except:
        print("config_video.json not found")
        exit()
    # Initialize an IPv4 socket with TCP (default) and try to connect to the nn
    streamlit_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    streamlit_socket.connect((streamlit_tunnel_addr, streamlit_tunnel_port))
    streamlit_live_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    streamlit_live_socket.connect((streamlit_live_tunnel_addr, streamlit_live_tunnel_port))
    return streamlit_socket, streamlit_live_socket, config

def start_bo(config, streamlit_socket, unique_class_labels, received_images_reshaped, received_labels_decoded, received_images_reshaped_test, received_labels_decoded_test, dataset_shape, ping_thread, event):
    global call_counter
    call_counter = 0
    global extra_results
    extra_results = []
    # Get unique labels in our (received) training dataset
    unique_class_labels = np.unique(received_labels_decoded)
    for i in range(1):
        # Get the new number of epochs from the keyboard
        data, my_stats = bo_res(unique_class_labels, received_images_reshaped, received_labels_decoded, received_images_reshaped_test, received_labels_decoded_test, dataset_shape)
        event.set()
        ping_thread.join()
        serialized_df = pickle.dumps(my_stats)
        streamlit_socket.sendall(len(serialized_df).to_bytes(8, 'big'))
        serialized_df = pickle.dumps(my_stats)
        streamlit_socket.sendall(serialized_df)
    # # # Close socket on exit
    # streamlit_socket.close()
    # streamlit_live_socket.close()
    # print("Sockets Closed")

# @use_named_args(dimensions = dimensions)
def fitness(*args):
    args = args[0]
    sample_size, epochs_number, conv_number, pool_number, dense_number, lstm_number, gru_number, rnn_number, dask_workers = [0, 0, 0, 0, 0, 0, 0, 0, 0]
    for i, dim_name in enumerate(dimension_names):
      if dim_name=='sample_size':
        sample_size = args[i]
      elif dim_name=='epochs_number':
        epochs_number = args[i]
      elif dim_name=='conv_number':
        conv_number = args[i]
      elif dim_name=='pool_number':
        pool_number = args[i]
      elif dim_name=='dense_number':
        dense_number = args[i]
      elif dim_name=='lstm_number':
        lstm_number = args[i]
      elif dim_name=='gru_number':
        gru_number = args[i]
      elif dim_name=='rnn_number':
        rnn_number = args[i]
      elif dim_name=='dask_workers':
        dask_workers = args[i]
    print()
    print(f"EPOCHS to: {epochs_number} \n SAMPLE RATE to: {sample_size} \n NUM_OF_CONV_LAYERS to: {conv_number} \n NUM_OF_POOL_LAYERS to: {pool_number} \n NUM_OF_DENSE_LAYERS to: {dense_number} \n NUM_OF_LSTM_LAYERS to: {lstm_number} \n NUM_OF_GRU_LAYERS to: {gru_number} \n NUM_OF_RNN_LAYERS to: {rnn_number} \n NUM_OF_WORKERS to: {dask_workers}")
    print()
    global call_counter
    call_counter += 1
    input_str = f"CALL: {call_counter}/{bo_call_number} \n EPOCHS to: {epochs_number} \n SAMPLE RATE to: {sample_size} \n NUM_OF_CONV_LAYERS to: {conv_number} \n NUM_OF_POOL_LAYERS to: {pool_number} \n NUM_OF_DENSE_LAYERS to: {dense_number}\n NUM_OF_LSTM_LAYERS to: {lstm_number} \n NUM_OF_GRU_LAYERS to: {gru_number} \n NUM_OF_RNN_LAYERS to: {rnn_number} \n NUM_OF_WORKERS to: {dask_workers}\n"
    # Call the sampling method
    cluster.scale(dask_workers)
    start_t = time.time()
    x_train, y_train = sampling_method(sampling_method_id, received_images_reshaped, received_labels_decoded, sample_size, dask_workers)
    print(f"Sampling needed {time.time()-start_t} seconds")
    layers_lst = []
    if conv_number > 0:
      if conv_number > pool_number:
        for i in range(0, conv_number - pool_number):
          layers_lst.append('conv')
        for i in range(0, pool_number):
          layers_lst.append('conv')
          layers_lst.append('pool')
      elif conv_number == pool_number:
        for i in range(0, conv_number):
          layers_lst.append('conv')
          layers_lst.append('pool')
      else:
        for i in range(0, conv_number):
          layers_lst.append('conv')
          layers_lst.append('pool')
        for i in range(conv_number, pool_number):
          layers_lst.append('pool')
    if lstm_number > 0:
      for i in range(0, lstm_number):
        layers_lst.append('lstm')
    if gru_number > 0:
      for i in range(0, gru_number):
        layers_lst.append('gru')
    if rnn_number > 0:
      for i in range(0, rnn_number):
        layers_lst.append('rnn')
    if dense_number > 0:
      for i in range(0, dense_number):
        layers_lst.append('dense')
    print(f"---------------------------->{layers_lst}")
    q = Queue()
    process_eval = multiprocessing.Process(target=my_evaluate, args=(q, x_train, y_train, received_images_reshaped_test, received_labels_decoded_test, layers_lst, epochs_number, lr, size_of_batch, CONV_NEURONS_CONST, UNITS_CONST, DENSE_NEURONS_CONST, CONV_NEURONS_BOUND, UNITS_BOUND, DENSE_NEURONS_BOUND, input_str))
    process_eval.start()
    test_acc, tr_time, train_loss, train_acc, tradeOff_metric = q.get()
    process_eval.join()


    # Print the results.
    print()
    print("Accuracy (on the testing dataset): {0:.2%}".format(test_acc))
    print(f"Training time: ", tr_time)
    print(tradeOff_metric)
    print()
    # Store the accuracy and the training speed of the corresponding model in order to be printed in the final cell
    tmp = [test_acc, tr_time, train_loss, train_acc]
    extra_results.append(tmp)
    # Delete the Keras model with these hyper-parameters from memory.
    K.clear_session()
    gc.collect()
    del x_train
    del y_train
    # Clear the Keras session, otherwise it will keep adding new
    # models to the same TensorFlow graph each time we create
    # a model with a different set of hyper-parameters.
    tf.compat.v1.reset_default_graph()
    return -tradeOff_metric


def bo_res(unique_class_labels, received_images_reshaped, received_labels_decoded, received_images_reshaped_test, received_labels_decoded_test, dataset_shape):

    gp_result = gp_minimize(func=fitness,
                            dimensions = dimensions,
                            n_calls = bo_call_number,
                            acq_func = acquisition_f,
                            noise = "gaussian",
                            n_jobs = -1,
                            x0 = default_parameters)
    all_cols = ['Sample Size', 'Epochs', 'Conv', 'Pool', 'Dense', 'LSTM', 'GRU', 'RNN', "Dask Workers"]
    df_extra = pd.DataFrame(extra_results, columns=["Accuracy", "Training Speed (sec)", "Loss Epoch", "Acc Epoch"])
    df_zero = pd.DataFrame(np.zeros((bo_call_number, len(all_cols)), dtype=int), columns=all_cols)
    df_res = pd.DataFrame(gp_result.x_iters, columns=column_names)
    for col in df_zero.columns:
      if col in df_res.columns:
        df_zero[col] = df_res[col]
    print(df_res)
    print(df_zero)
    pd_tmp = pd.concat([df_zero, (pd.Series(gp_result.func_vals * -1, name="Score"))], axis=1)
    final_result = pd.concat([pd_tmp, df_extra], axis=1)
    final_result['Sample Size'] = final_result['Sample Size'].round(3)

    # print(gp_result.x)
    all_dim_names = ['sample_size', 'epochs_number', 'conv_number', 'pool_number', 'dense_number', 'lstm_number', 'gru_number', 'rnn_number', 'dask_workers']
    result = ['0', '0', '0', '0', '0', '0', '0', '0', '0']
    for i, dim_name in enumerate(dimension_names):
      result[all_dim_names.index(dim_name)] = gp_result.x[i].__str__()

    print(f" NEW EPOCHS to: {result[1]} \n NEW SAMPLE RATE to: {result[0]} \n NUM_OF_CONV_LAYERS to: {result[2]} \n NUM_OF_POOL_LAYERS to: {result[3]} \n NUM_OF_DENSE_LAYERS to: {result[4]} \n NUM_OF_LSTM_LAYERS to: {result[5]} \n NUM_OF_GRU_LAYERS to: {result[6]} \n NUM_OF_RNN_LAYERS to: {result[7]} \n NUM_OF_WORKERS to: {result[8]}")

    tmp = result[1] +','+result[0]+','+result[2]+','+result[3]+','+result[4]+','+result[5]+','+result[6]+','+result[7]+','+result[8]

    return tmp, final_result

def sampling_method(sampling_method_id, received_images_reshaped, received_labels_decoded, sample_size, n_workers):
  # print("Percentage of filtering in our training dataset was set:")
  # print(sample_size)

  if sampling_method_id == 0:
    # Simple reservoir sampling over the whole training dataset
    # Total size of the stream (or training dataset)
    n_train = len(received_images_reshaped)

    # Number of samples that will be drawn
    k_train = int(n_train * sample_size)

    # Use the indexes of dataset in order to decide which samples will be drawn
    idx_tmp_train_list = list(range(0, n_train))

    # Find the indexes in order to construct the dataset that will be used during the training process
    idx_train = reservoir_sampling(idx_tmp_train_list, n_train, k_train)
  elif sampling_method_id == 1:
    # Reservoir sampling in each class based on the number of samples (per class) that exist in the initial dataset
    # Find the size of each reservoir for every class depending on its occurence in the initial training dataset
    class_perc, unique_ids = reservoir_size_per_class(received_labels_decoded)

    # Stores the indexes (from all classes) in order to construct the dataset that will be used during the training process
    idx_train = []

    # Run for every single class the reservoir sampling seperately
    for i in range(0, len(unique_ids)):
      # Find the locations of each sample belonging to our class of interest
      tmp = np.where(np.asarray(received_labels_decoded) == unique_ids[i])
      idx_of_class = tmp[0].tolist()

      # Run the reservoir sampling for the class of interest
      sampled_idx_of_class = reservoir_sampling(idx_of_class, len(idx_of_class), int(len(received_images_reshaped) * sample_size * class_perc[i]))

      # Store the (sampled) samples from this class
      for j in range(0, len(sampled_idx_of_class)):
        idx_train.append(sampled_idx_of_class[j])
  elif sampling_method_id == 2:
      # Reservoir sampling in each class based on the number of samples (per class) that exist in the initial dataset
      # Find the size of each reservoir for every class depending on its occurence in the initial training dataset
      class_perc, unique_ids = reservoir_size_per_class_da(received_labels_decoded)
      # Stores the indexes (from all classes) in order to construct the dataset that will be used during the training process
      idx_train = []
      tmp = da.where((received_labels_decoded >= 0) & (received_labels_decoded <= np.max(unique_ids)))
      idx_of_class = tmp[0].compute_chunk_sizes()
      k = int(len(received_labels_decoded) * sample_size)
      PS = PrioritySampler(k)
      sampled_idx_of_class = PS.reservoir_priority_sampling(all_data = idx_of_class, k = k, divided_by = 1)
      idx_train.extend(sampled_idx_of_class)
      train_images = received_images_reshaped[idx_train].compute()
      train_labels = received_labels_decoded[idx_train].compute()
      print(np.bincount(train_labels[:, 0]))
      return train_images, train_labels

      # Run for every single class the reservoir sampling seperately
      # for i in range(0, len(unique_ids)):
      #   # Find the locations of each sample belonging to our class of interest
      #   tmp = da.where(received_labels_decoded == unique_ids[i])
      #   # idx_of_class = tmp[0].tolist()
      #   idx_of_class = tmp[0].compute_chunk_sizes()

      #   k = int(len(received_images_reshaped) * sample_size * class_perc[i])
      #   PS = PrioritySampler(k)
      #   # Run the reservoir sampling for the class of interest
      #   st = time.time()
      #   sampled_idx_of_class = PS.reservoir_priority_sampling(all_data=idx_of_class, k=k, divided_by=4)
      #   print(time.time()-st)
      #   # Store the (sampled) samples from this class
      #   idx_train.extend(sampled_idx_of_class)

  # Store the corresponding images and labels from training dataset based on the sampled indexes
  train_images_lst = []
  for i in idx_train:
      train_images_lst.append(received_images_reshaped[i])

  train_labels_lst = []
  for i in idx_train:
      train_labels_lst.append(received_labels_decoded[i])

  # Check the occurrence of each class in the final training dataset
  # print_times_per_label(train_labels_lst, received_labels_decoded)

  # Transform the lists that we stored our samples into arrays
  train_images = np.asarray(train_images_lst)
  train_labels = np.asarray(train_labels_lst)
  # train_images_lst = []
  # for i in idx_train:
  #   train_images_lst.append(received_images_reshaped[i])

  # train_labels_lst = []
  # for i in idx_train:
  #   train_labels_lst.append(received_labels_decoded[i])

  # # Check the occurence of each class in the final training dataset
  # # print_times_per_label(train_labels_lst, received_labels_decoded)

  # # Tranfsorm the lists that we stored our samples into arrays
  # train_images = np.asarray(train_images_lst)
  # train_labels = np.asarray(train_labels_lst)

  # Verify that the desired filtering was performed in both datasets
  # print("Training dataset before sampling:")
  # print(len(received_images_reshaped))
  # print(len(received_labels_decoded))
  # print("Training dataset after sampling:")

  return train_images, train_labels

# A function to randomly select k items from stream[0..n-1].
def reservoir_sampling(stream, n, k):
    i = 0  # index for elements in stream[]

    # reservoir[] is the output array.
    # Initialize it with first k elements from stream[]
    reservoir = [0] * k

    for i in range(k):
        reservoir[i] = stream[i]

    # Iterate from the (k+1)th element to Nth element
    while (i < n):
        # Pick a random index from 0 to i.
        j = random.randrange(i + 1)

        # If the randomly picked
        # index is smaller than k,
        # then replace the element
        # present at the index
        # with new element from stream
        if (j < k):
            reservoir[j] = stream[i]
        i += 1
    return reservoir

def reservoir_size_per_class(init_labels):
    # Get unique labels and their counts (how many times they appear) in our training dataset
    unique_labels, counts = np.unique(init_labels, return_counts=True)

    # Transform to list
    unique_labels_lst = unique_labels.tolist()
    counts_lst = counts.tolist()

    perc_per_class = []
    for i in range(len(unique_labels_lst)):
        perc_per_class.append(counts_lst[i] / len(init_labels))

    # print(perc_per_class)

    return perc_per_class, unique_labels_lst

# A function that finds the size of each reservoir for every class depending on its occurence in the initial dataset
# and returns the unique labels that exist in our dataset along with the corresponding percentage
def reservoir_size_per_class_da(init_labels):
  # Get unique labels and their counts (how many times they appear) in our training dataset

  # Compute unique labels
  unique_labels = da.unique(init_labels).compute()

  # Use bincount to get the counts of each unique label
  counts = da.bincount(init_labels.ravel()).compute()

  # Now, filter the counts to include only those for the unique labels
  counts = counts[unique_labels]

  # Transform to list
  unique_labels_lst = unique_labels.tolist()
  counts_lst = counts.tolist()

  perc_per_class = []
  for i in range(len(unique_labels_lst)):
    perc_per_class.append(counts_lst[i] / len(init_labels))

  return perc_per_class, unique_labels_lst


# A function that prints the occurence of each class in a list
def print_times_per_label(lst, labels_all):
  # Get unique labels in our training dataset
  unique_labels = np.unique(labels_all)
  for i in range(0, len(unique_labels)):
    print("Class", unique_labels[i], "has", lst.count(i), "samples in our dataset...")



def my_evaluate(q, x_train, y_train, features_test, labels_test, layers_lst, epochs_number, learning_rate, batch_size, CONV_NEURONS_CONST, UNITS_CONST, DENSE_NEURONS_CONST, CONV_NEURONS_BOUND, UNITS_BOUND, DENSE_NEURONS_BOUND, input_str):
  error_flag = -1

  try:
    # Function that creates the model
    model, *_ = create_model(layers_lst, 'dense', CONV_NEURONS_CONST, UNITS_CONST, DENSE_NEURONS_CONST, CONV_NEURONS_BOUND, UNITS_BOUND, DENSE_NEURONS_BOUND)

    if model == -1:
      return -1000000

    # If the just-added-layer was conv or pool then add manually a flatten layer
    if 'lstm' not in layers_lst and 'gru' not in layers_lst and 'rnn' not in layers_lst and 'dense' not in layers_lst:
      model.add(Flatten())

    # Softmax is an activation function that is used mainly for classification tasks
    # It normalizes the input vector into a probability distribution  that is proportional to the exponential of the input numbers.
    model.add(tf.keras.layers.Dense(len(UNIQUE_CLASS_LABELS), activation = "softmax"))
  except ValueError:
    print("No valid input...:(")
    error_flag = 1

  if error_flag == -1:
    model.compile(optimizer=Adam(learning_rate),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

    stringlist = []
    model.summary(print_fn=lambda x: stringlist.append(x))
    short_model_summary = "\n".join(stringlist)

    # Here we reshape the input of the network based on the type of the first layer of the network
    # If the first layer is conv
    if layers_lst[0] == 'conv':
      reshaped_x_train = x_train
      reshaped_x_test = features_test
    # If the first layer is lstm-gru-rnn
    elif layers_lst[0] == 'lstm' or layers_lst[0] == 'gru' or layers_lst[0] == 'rnn':
      num_samples, num_frames, height, width, channels = x_train.shape
      reshaped_x_train = x_train.reshape(num_samples, num_frames, height * width * channels)
      num_samples, num_frames, height, width, channels = features_test.shape
      reshaped_x_test = features_test.reshape(num_samples, num_frames, height * width * channels)
    # If the first layer is dense
    else:
      num_samples, num_frames, height, width, channels = x_train.shape
      reshaped_x_train = x_train.reshape(num_samples, num_frames * height * width * channels)
      num_samples, num_frames, height, width, channels = features_test.shape
      reshaped_x_test = features_test.reshape(num_samples, num_frames * height * width * channels)
    start = time.time()
    np.int = int
    blackbox = model.fit(x=reshaped_x_train,
                        y=y_train,
                        epochs=epochs_number,
                        batch_size=batch_size
                        )
    stop = time.time()
    tr_loss_lst = blackbox.history['loss']
    tr_accuracy_lst = blackbox.history['accuracy']
    # Compute the training speed of this CNN architecture
    tr_time = stop - start

    # Compute the accuracy of our training model in the testing dataset
    test_loss, test_acc = model.evaluate(reshaped_x_test,  labels_test, verbose=2)

    # # Return the validation accuracy for the last epoch.
    # accuracy = blackbox.history['val_accuracy'][-1]

    # Compute the metric that captures the accuracy--speed tradeoff
    tradeOff_metric = lamda_acc * test_acc - (1 - lamda_acc) * math.tanh(tr_time/theta_parameter - 1)

    # Print the results.
    print()
    print("Accuracy (on the testing dataset): {0:.2%}".format(test_acc))
    print(f"Training time: ", tr_time)
    print(tradeOff_metric)
    print()

    tmp = "\nAccuracy (on the testing dataset): {0:.2%}".format(
        test_acc) + '\nTraining time:' + tr_time.__str__() + '\nTradeOff Metric:' + tradeOff_metric.__str__() + '\n\n'
    msg = pickle.dumps(input_str + short_model_summary + tmp)
    data_length = struct.pack('!I', len(msg))  # !I is for network order and unsigned int (4 bytes)

    streamlit_live_socket.sendall(data_length)
    streamlit_live_socket.sendall(msg)

    # Delete the Keras model with these hyper-parameters from memory.
    del model
    q.put([test_acc, tr_time, tr_loss_lst, tr_accuracy_lst, tradeOff_metric])
  else:
    q.put([0, 1000000000, 1000000000, 0, 0])


def create_model(layers_lst, layer2add, CONV_NEURONS_CONST, UNITS_CONST, DENSE_NEURONS_CONST, CONV_NEURONS_BOUND,
                 UNITS_BOUND, DENSE_NEURONS_BOUND):

    if layers_lst[0] == 'pool' or len(layers_lst) == 0:
        return -1

    # Initialize a sequential model
    model = tf.keras.models.Sequential()

    # Define the number of neurons for conv and dense layers and the number of units for lstm-gru-rnn
    conv_tmp2 = CONV_NEURONS_CONST
    units_tmp2 = UNITS_CONST
    dense_tmp2 = DENSE_NEURONS_CONST

    # Find the type of the next and the previous layer because you need different configurations
    for count, layer in enumerate(layers_lst):
        if count == 0 and len(layers_lst) > 1:
            previous_layer_tmp = 'no'
            next_layer_tmp = layers_lst[count + 1]
        elif count == 0:
            previous_layer_tmp = 'no'
            next_layer_tmp = 'no'
        elif count == len(layers_lst) - 1:
            next_layer_tmp = 'no'
            previous_layer_tmp = layers_lst[count - 1]
        else:
            previous_layer_tmp = layers_lst[count - 1]
            next_layer_tmp = layers_lst[count + 1]

        # Recreate the so-far-model
        # First layer conv
        if layer == 'conv' and count == 0:
            model.add(TimeDistributed(Conv2D(int(conv_tmp2), (3, 3), padding='same', activation='relu'),
                                      input_shape=(SEQUENCE_LENGTH, DATASET_SHAPE[0], DATASET_SHAPE[1], 3)))
            conv_tmp2 = conv_tmp2 * 2
        # First layer lstm-gru-rnn (change the shape of the input) and next or 2-be-added layer lstm-gru-rnn (should add the 'return conf')
        elif ((layer == 'lstm' or layer == 'gru' or layer == 'rnn') and (((count == 0) and len(
            layers_lst) == 1 and (layer2add == 'lstm' or layer2add == 'gru' or layer2add == 'rnn')) or (
                                                                             (count == 0) and (
                                                                             next_layer_tmp == 'lstm' or next_layer_tmp == 'gru' or next_layer_tmp == 'rnn')))):
            if layer == 'lstm':
                model.add(tf.keras.layers.LSTM(int(units_tmp2), return_sequences=True,
                                               input_shape=(SEQUENCE_LENGTH, DATASET_SHAPE[0] * DATASET_SHAPE[1] * 3)))
            elif layer == 'gru':
                model.add(tf.keras.layers.GRU(int(units_tmp2), return_sequences=True,
                                              input_shape=(SEQUENCE_LENGTH, DATASET_SHAPE[0] * DATASET_SHAPE[1] * 3)))
            else:
                model.add(tf.keras.layers.SimpleRNN(int(units_tmp2), return_sequences=True,
                                                    input_shape=(SEQUENCE_LENGTH, DATASET_SHAPE[0] * DATASET_SHAPE[1] * 3)))
            units_tmp2 = units_tmp2 / 2
        # First layer lstm-gru-rnn (change the shape of the input)
        elif ((layer == 'lstm' or layer == 'gru' or layer == 'rnn') and count == 0):
            if layer == 'lstm':
                model.add(tf.keras.layers.LSTM(int(units_tmp2),
                                               input_shape=(SEQUENCE_LENGTH, DATASET_SHAPE[0] * DATASET_SHAPE[1] * 3)))
            elif layer == 'gru':
                model.add(tf.keras.layers.GRU(int(units_tmp2),
                                              input_shape=(SEQUENCE_LENGTH, DATASET_SHAPE[0] * DATASET_SHAPE[1] * 3)))
            else:
                model.add(tf.keras.layers.SimpleRNN(int(units_tmp2),
                                                    input_shape=(SEQUENCE_LENGTH, DATASET_SHAPE[0] * DATASET_SHAPE[1] * 3)))
            units_tmp2 = units_tmp2 / 2
        # First layer densse (change the shape of the input)
        elif layer == 'dense' and count == 0:
            model.add(tf.keras.layers.Dense(int(dense_tmp2), activation='relu',
                                            input_shape=(SEQUENCE_LENGTH * DATASET_SHAPE[0] * DATASET_SHAPE[1] * 3,)))
            dense_tmp2 = dense_tmp2 / 2
        # For the remaining layers
        else:
            if layer == 'conv':
                # Add a conv layer by doubling its neurons if they do not violate our user-defined bound
                if conv_tmp2 <= CONV_NEURONS_BOUND:
                    model.add(TimeDistributed(Conv2D(int(conv_tmp2), (3, 3), padding='same', activation='relu')))
                    conv_tmp2 = conv_tmp2 * 2
                else:
                    model.add(
                        TimeDistributed(Conv2D(int(CONV_NEURONS_BOUND), (3, 3), padding='same', activation='relu')))
                    conv_tmp2 = CONV_NEURONS_BOUND
            elif layer == 'pool':
                # Add a pool layer
                model.add(TimeDistributed(MaxPooling2D((4, 4))))
            elif layer == 'lstm':
                # If the previous layer is conv or pool add a flatten layer first
                if previous_layer_tmp == 'conv' or previous_layer_tmp == 'pool':
                    model.add(TimeDistributed(BatchNormalization()))
                    model.add(TimeDistributed(Flatten()))
                # Add a lstm layer by reducing (* 0.5) its units if they do not violate our user-defined bound
                if units_tmp2 >= UNITS_BOUND:
                    # If the next layer is dense then do not return sequences
                    if next_layer_tmp == 'dense' or (layer2add == 'dense' and count == len(layers_lst) - 1):
                        model.add(tf.keras.layers.LSTM(int(units_tmp2)))
                    else:
                        model.add(tf.keras.layers.LSTM(int(units_tmp2), return_sequences=True))
                    units_tmp2 = units_tmp2 / 2
                else:
                    # If the next layer is dense then do not return sequences
                    if next_layer_tmp == 'dense' or (layer2add == 'dense' and count == len(layers_lst) - 1):
                        model.add(tf.keras.layers.LSTM(int(UNITS_BOUND)))
                    else:
                        model.add(tf.keras.layers.LSTM(int(UNITS_BOUND), return_sequences=True))
                    units_tmp2 = UNITS_BOUND
            elif layer == 'gru':
                # If the previous layer is conv or pool add a flatten layer first
                if previous_layer_tmp == 'conv' or previous_layer_tmp == 'pool':
                    model.add(TimeDistributed(BatchNormalization()))
                    model.add(TimeDistributed(Flatten()))
                # Add a gru layer by reducing (* 0.5) its units if they do not violate our user-defined bound
                if units_tmp2 >= UNITS_BOUND:
                    # If the next layer is dense then do not return sequences
                    if next_layer_tmp == 'dense' or (layer2add == 'dense' and count == len(layers_lst) - 1):
                        model.add(tf.keras.layers.GRU(int(units_tmp2)))
                    else:
                        model.add(tf.keras.layers.GRU(int(units_tmp2), return_sequences=True))
                    units_tmp2 = units_tmp2 / 2
                else:
                    # If the next layer is dense then do not return sequences
                    if next_layer_tmp == 'dense' or (layer2add == 'dense' and count == len(layers_lst) - 1):
                        model.add(tf.keras.layers.GRU(int(UNITS_BOUND)))
                    else:
                        model.add(tf.keras.layers.GRU(int(UNITS_BOUND), return_sequences=True))
                    units_tmp2 = UNITS_BOUND
            elif layer == 'rnn':
                # If the previous layer is conv or pool add a flatten layer first
                if previous_layer_tmp == 'conv' or previous_layer_tmp == 'pool':
                    model.add(TimeDistributed(BatchNormalization()))
                    model.add(TimeDistributed(Flatten()))
                # Add a rnn layer by reducing (* 0.5) its units if they do not violate our user-defined bound
                if units_tmp2 >= UNITS_BOUND:
                    # If the next layer is dense then do not return sequences
                    if next_layer_tmp == 'dense' or (layer2add == 'dense' and count == len(layers_lst) - 1):
                        model.add(tf.keras.layers.SimpleRNN(int(units_tmp2)))
                    else:
                        model.add(tf.keras.layers.SimpleRNN(int(units_tmp2), return_sequences=True))
                    units_tmp2 = units_tmp2 / 2
                else:
                    # If the next layer is dense then do not return sequences
                    if next_layer_tmp == 'dense' or (layer2add == 'dense' and count == len(layers_lst) - 1):
                        model.add(tf.keras.layers.SimpleRNN(int(UNITS_BOUND)))
                    else:
                        model.add(tf.keras.layers.SimpleRNN(int(UNITS_BOUND), return_sequences=True))
                    units_tmp2 = UNITS_BOUND
            else:
                if previous_layer_tmp == 'conv' or previous_layer_tmp == 'pool':
                    model.add(Flatten())
                # Add a dense layer by reducing (* 0.5) its neurons if they do not violate our user-defined bound
                if dense_tmp2 >= DENSE_NEURONS_BOUND:
                    model.add(tf.keras.layers.Dense(int(dense_tmp2), activation='relu'))
                    dense_tmp2 = dense_tmp2 / 2
                else:
                    model.add(tf.keras.layers.Dense(int(DENSE_NEURONS_BOUND), activation='relu'))
                    dense_tmp2 = DENSE_NEURONS_BOUND

    return model, conv_tmp2, units_tmp2, dense_tmp2

def run_subito_opt():
    streamlit_socket, streamlit_live_socket_tmp, config = start_controller()
    global streamlit_live_socket
    streamlit_live_socket = streamlit_live_socket_tmp
    event = Event()
    ping_thread = Thread(target=ping_socket, args=(streamlit_socket, event,))
    ping_thread.start()
    start_bo(config, streamlit_socket, UNIQUE_CLASS_LABELS, received_images_reshaped, received_labels_decoded, received_images_reshaped_test, received_labels_decoded_test, DATASET_SHAPE, ping_thread, event)
    streamlit_socket.close()
    streamlit_live_socket.close()
    print("Sockets Closed")
    print("Returned")

def ping_socket(socket, event):
    while True:
        if event.is_set():
            return
        socket.sendall(int(1).to_bytes(8, 'big'))
        time.sleep(10)

def socket_listener(conn):
    print("Waiting for run signal...")
    while True:
        # Receive data from the socket
        try:
            data = conn.recv(10)

            print('raw data is', data)
            data = data.decode()
            print('decoded data is', data)
            if data == 'start':
                print("Received run signal")
                run_subito_opt()
                print("--------------Go subito Go--------------")
        except Exception as error:
            print("An exception occurred:", error)
            print("Disconnecting Socket")
            #run_socket.connect((sbto_run_tunnel_addr, sbto_run_tunnel_port))

if __name__ == "__main__":
    import warnings
    warnings.filterwarnings("ignore")
    try:
        with open('config_video.json') as json_file:
            config = json.load(json_file)
    except:
        print("config_video.json not found")
        exit()
    (train_images_all, train_labels_all), (test_images_all, test_labels_all) = (x_train, y_train), (x_test, y_test)
    train_images_all, test_images_all = train_images_all / 255.0, test_images_all / 255.0
    received_images_reshaped = train_images_all
    received_labels_decoded = train_labels_all
    received_images_reshaped_test = test_images_all
    received_labels_decoded_test = test_labels_all

    st = time.time()
    train_images, train_labels = sampling_method(sampling_method_id, received_images_reshaped, received_labels_decoded, 0.2, 4)
    print(time.time()-st)
    print(len(train_images))
    print(len(train_labels))

    print("Received Training Data:")
    print("------> # of received videos:", len(received_images_reshaped))
    print("------> # of received labels:", len(received_labels_decoded))
    print("Received Testing Data:")
    print("------> # of received video:", len(received_images_reshaped_test))
    print("------> # of received labels:", len(received_labels_decoded_test))

    # Initialize an IPv4 socket with TCP (default) and try to connect to the nn
    run_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    run_socket.connect((sbto_run_tunnel_addr, sbto_run_tunnel_port))
    socket_listener(run_socket)

[18 14 23 17 17 22 25 22 23 19]
1.7189702987670898
200
200
Received Training Data:
------> # of received videos: 1003
------> # of received labels: 1003
Received Testing Data:
------> # of received video: 335
------> # of received labels: 335
Waiting for run signal...
raw data is b'p'
decoded data is p
raw data is b'start'
decoded data is start
Received run signal


INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:45875'



EPOCHS to: 5 
 SAMPLE RATE to: 0.15 
 NUM_OF_CONV_LAYERS to: 2 
 NUM_OF_POOL_LAYERS to: 2 
 NUM_OF_DENSE_LAYERS to: 2 
 NUM_OF_LSTM_LAYERS to: 1 
 NUM_OF_GRU_LAYERS to: 0 
 NUM_OF_RNN_LAYERS to: 0 
 NUM_OF_WORKERS to: 4



INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:42833'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:39045 name: 2
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:39045
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:38356
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:39831 name: 3
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:39831
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:38362


[14 17 21 10 11 13 17 15  8 24]
Sampling needed 2.539024591445923 seconds
---------------------------->['conv', 'pool', 'conv', 'pool', 'lstm', 'dense', 'dense']
Epoch 1/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 11s 26ms/step - accuracy: 0.2140 - loss: 2.2782
Epoch 2/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.4205 - loss: 1.9538
Epoch 3/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.5013 - loss: 1.6130
Epoch 4/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.5880 - loss: 1.3319
Epoch 5/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.7255 - loss: 1.0843
11/11 - 2s - 144ms/step - accuracy: 0.4806 - loss: 1.6895

Accuracy (on the testing dataset): 48.06%
Training time:  12.488057374954224
0.2994988368992926


Accuracy (on the testing dataset): 48.06%
Training time:  12.488057374954224
0.2994988368992926



...


EPOCHS to: 11 
 SAMPLE RATE to: 0.12874400170483905 
 NUM_OF_CONV_LAYERS to: 1 
 NUM_OF_POOL_LAYERS to: 1 
 NUM_OF_DENSE_LAYERS to: 1 
 NUM_OF_LSTM_LAYERS to: 1 
 NUM_OF_GRU_LAYERS to: 1 
 NUM_OF_RNN_LAYERS to: 0 
 NUM_OF_WORKERS to: 6



INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:39789'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:36849 name: 9
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:36849
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:42244
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:42643 name: 10
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:42643
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:42256


[11  7 21  8  5 25 12  9 20 11]
Sampling needed 2.3102564811706543 seconds
---------------------------->['conv', 'pool', 'lstm', 'gru', 'dense']
Epoch 1/11
13/13 ━━━━━━━━━━━━━━━━━━━━ 10s 28ms/step - accuracy: 0.1739 - loss: 2.2497
Epoch 2/11
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.5629 - loss: 1.8286
Epoch 3/11
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.5525 - loss: 1.4236
Epoch 4/11
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.7096 - loss: 1.0660
Epoch 5/11
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.7504 - loss: 0.8928
Epoch 6/11
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.8814 - loss: 0.6470
Epoch 7/11
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.8842 - loss: 0.5170
Epoch 8/11
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.9518 - loss: 0.4084
Epoch 9/11
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.9054 - loss: 0.4679
Epoch 10/11
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.9696 - loss: 0.2819
Epo

KeyboardInterrupt: 

INFO:distributed.nanny:Closing Nanny gracefully at 'tcp://127.0.0.1:33595'. Reason: worker-close
INFO:distributed.nanny:Closing Nanny gracefully at 'tcp://127.0.0.1:39789'. Reason: worker-close
INFO:distributed.nanny:Closing Nanny gracefully at 'tcp://127.0.0.1:39069'. Reason: worker-close
INFO:distributed.nanny:Closing Nanny gracefully at 'tcp://127.0.0.1:45965'. Reason: worker-close


In [19]:
client.shutdown()

INFO:distributed.scheduler:Retire worker addresses (stimulus_id='retire-workers-1753704781.4193778') (0, 1)
INFO:distributed.nanny:Closing Nanny at 'tcp://127.0.0.1:34747'. Reason: nanny-close
INFO:distributed.nanny:Nanny asking worker to close. Reason: nanny-close
INFO:distributed.nanny:Closing Nanny at 'tcp://127.0.0.1:46335'. Reason: nanny-close
INFO:distributed.nanny:Nanny asking worker to close. Reason: nanny-close
INFO:distributed.core:Received 'close-stream' from tcp://127.0.0.1:58338; closing.
INFO:distributed.scheduler:Remove worker addr: tcp://127.0.0.1:37801 name: 0 (stimulus_id='handle-worker-cleanup-1753704781.4254146')
INFO:distributed.core:Received 'close-stream' from tcp://127.0.0.1:58322; closing.
INFO:distributed.scheduler:Remove worker addr: tcp://127.0.0.1:34581 name: 1 (stimulus_id='handle-worker-cleanup-1753704781.4282067')
INFO:distributed.scheduler:Lost all workers
INFO:distributed.nanny:Nanny at 'tcp://127.0.0.1:46335' closed.
INFO:distributed.nanny:Nanny at 't

In [20]:
import dask
from dask.distributed import Client, LocalCluster

n_workers = 2
thread_per_worker = 1
dask.config.set(scheduler='threads', num_of_workers=n_workers, threads_per_worker=thread_per_worker)
cluster = LocalCluster(n_workers=n_workers, threads_per_worker=thread_per_worker, dashboard_address=':8888')
client = Client(cluster)
print(f'{cluster.dashboard_link}')
import copy
x_train = copy.deepcopy(tmp1)
y_train = copy.deepcopy(tmp2)
x_test = copy.deepcopy(tmp3)
y_test = copy.deepcopy(tmp4)
sampling_method_id = 2
import dask.array as da
print(f"train images len: {x_train.shape}")
print(f"train labels len: {y_train.shape}")
y_train = y_train.reshape(-1, 1)
divided_by = 4

rep_factor = 1

if sampling_method_id == 2:
  x_train = da.from_array(np.tile(x_train, (rep_factor,1,1,1,1)), chunks=((len(x_train) * rep_factor) // divided_by, 20, 64 ,64 ,3))  # You can adjust the chunk size as needed
  y_train = da.from_array(np.tile(y_train, (rep_factor,1)), chunks=((len(y_train) * rep_factor) // divided_by, 1))  # You can adjust the chunk size as needed
else:
  x_train = np.tile(x_train, (rep_factor,1,1,1))  # You can adjust the chunk size as needed
  y_train = np.tile(y_train, (rep_factor,1))  # You can adjust the chunk size as needed

print(f"train labels: {y_train}")
print(f"train images: {x_train}")

# Compute unique labels
unique_class_labels = da.unique(y_train).compute()
print(x_train.shape)
print(y_train.shape)
#unique_class_labels = np.unique(train_labels_all)
# x_train, x_test = x_train / 255.0, x_test / 255.0



INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:41351
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:8888/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:39069'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:33595'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:42843 name: 1
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:42843
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:60812
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:43145 name: 0
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:43145
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:60822
INFO:distributed.scheduler:Receive client connection: Client-36adebcd-6bac-11f0-8347-0242ac1c000c
INFO:distributed.core:Starting establish

http://127.0.0.1:8888/status
train images len: (1003, 20, 64, 64, 3)
train labels len: (1003,)
train labels: dask.array<array, shape=(1003, 1), dtype=int64, chunksize=(250, 1), chunktype=numpy.ndarray>
train images: dask.array<array, shape=(1003, 20, 64, 64, 3), dtype=uint8, chunksize=(250, 20, 64, 64, 3), chunktype=numpy.ndarray>
(1003, 20, 64, 64, 3)
(1003, 1)
